In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

# Dataset

The data set is derived from the [MovieLens "ml-latest-small"](https://grouplens.org/datasets/movielens/latest/) dataset.   
[F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

The original dataset has  9000 movies rated by 600 users. The dataset has been reduced in size to focus on movies from the years since 2000. This dataset consists of ratings on a scale of 0.5 to 5 in 0.5 step increments. The reduced dataset has $n_u = 443$ users, and $n_m= 4778$ movies. 

Below, you will load the movie dataset into the variables $Y$ and $R$.

The matrix $Y$ (a  $n_m \times n_u$ matrix) stores the ratings $y^{(i,j)}$. The matrix $R$ is an binary-valued indicator matrix, where $R(i,j) = 1$ if user $j$ gave a rating to movie $i$, and $R(i,j)=0$ otherwise. 

In [2]:
df_y = pd.read_csv('./data/Y_MovieLens.csv', header=None)
df_y

,0,1,2,3,4,5,6,7,8,9,...,433,434,435,436,437,438,439,440,441,442
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,4.0,3.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4773,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4774,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4775,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4776,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
df_r = pd.read_csv('./data/R_MovieLens.csv', header=None)
df_r

,0,1,2,3,4,5,6,7,8,9,...,433,434,435,436,437,438,439,440,441,442
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4773,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4774,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4775,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4776,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Cost Function for Collaborative Filtering

$$J({\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},\mathbf{w}^{(0)},b^{(0)},...,\mathbf{w}^{(n_u-1)},b^{(n_u-1)}})$$ 

$$
= \left[ \frac{1}{2}\sum_{(i,j):r(i,j)=1}(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2 \right]
+ \underbrace{\left[
\frac{\lambda}{2}
\sum_{j=0}^{n_u-1}\sum_{k=0}^{n-1}(\mathbf{w}^{(j)}_k)^2
+ \frac{\lambda}{2}\sum_{i=0}^{n_m-1}\sum_{k=0}^{n-1}(\mathbf{x}_k^{(i)})^2
\right]}_{regularization}
$$

$$
= \left[ \frac{1}{2}\sum_{j=0}^{n_u-1} \sum_{i=0}^{n_m-1}r(i,j)*(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2 \right]
+\text{regularization}
$$

In [4]:
Y = df_y.values
R = df_r.values

print("Shape of Y:", Y.shape)
print("Shape of R:", R.shape)

Shape of Y: (4778, 443)
Shape of R: (4778, 443)


In [5]:
n_movies, n_users = df_y.shape
n_features = 10  # randomly

# Create the arrays with the specified shapes and conditions
X = np.random.rand(n_movies, n_features)  # Shape (4778, 10), values in range [0, 1]
W = np.random.rand(n_users, n_features)   # Shape (443, 10), values in range [0, 1]
B = np.zeros(n_users)                     # Shape (443,), all values are 0

# Display the shapes and first few values
print("Shape of X:", X.shape)
print("Shape of W:", W.shape)
print("Shape of B:", B.shape)

Shape of X: (4778, 10)
Shape of W: (443, 10)
Shape of B: (443,)


In [6]:
def compute_cost(X, W, B, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    
    Args:
      X -- ndarray (num_movies, num_features): matrix of item features
      W -- ndarray (num_users, num_features) : matrix of user parameters
      B -- ndarray (num_users, )             : vector of user parameters
      Y -- ndarray (num_movies, num_users)   : matrix of user ratings of movies
      R -- ndarray (num_movies, num_users)   : matrix, where R(i, j) = 1 if the j-th user rated the i-th movie
      lambda_ (float): regularization parameter
      
    Returns:
      J (float): Cost
    """
    
    n_items, n_users = Y.shape
    J = 0
    
    for j in range(n_users):
        w = W[j, :]  # Pick parameters w of the user j
        b = B[j]     # Pick parameter b of the user j
        
        for i in range(n_items):
            x = X[i, :]  # Pick parameters of the movie i
            
            r = R[i, j]  # Whether user j rated movie i or not
            y = Y[i, j]  # Pick the rating of user j gave on the movie 
            
            J += (1/2) * np.square(r * (np.dot(w,x) + b - y))  # error
            
    J += (lambda_/2) * (np.sum(np.square(W)) + np.sum(np.square(X)))  # regulalization

    return J

In [7]:
def compute_cost_vectorized(X, W, B, Y, R, lambda_):
    """
    Vectorization for speed. Uses tensorflow operations to be compatible with custom training loops.
    """
    # (n_movies, n_features) @ (n_users, n_features).T + (1, n_users) = (n_movies, n_users)
    
    error = R * (tf.linalg.matmul(X, tf.transpose(W)) + B - Y)  # B must have the shape of (1, n_users)
    mean_squared_error = (1/2) * tf.reduce_sum(error**2)
                      
    regularization = (lambda_/2) * (tf.reduce_sum(X**2) + tf.reduce_sum(W**2))
                  
    J = mean_squared_error + regularization
    return J

In [8]:
import time

# Measure the time for the non-vectorized cost function
start_time = time.time()
J_non_vectorized = compute_cost(X, W, B, Y, R, 1.5)
end_time = time.time()
print("Non-vectorized cost function time:", end_time - start_time, "seconds")

# Measure the time for the vectorized cost function
start_time = time.time()
B_reshaped = B.reshape(1, -1)
J_vectorized = compute_cost_vectorized(X, W, B, Y, R, 1.5)
end_time = time.time()
print("Vectorized cost function time:", end_time - start_time, "seconds")

print()
print(J_non_vectorized)
print(J_vectorized)

Non-vectorized cost function time: 11.953410863876343 seconds
Vectorized cost function time: 0.17380976676940918 seconds

61040.34915457633
tf.Tensor(61040.34915457647, shape=(), dtype=float64)


# Add new User

In [9]:
df_ratings = pd.read_csv('./data/movie_ratings.csv', index_col=0)
df_ratings

,mean rating,number of ratings,title
0,3.400000,5,"Yards, The (2000)"
1,3.250000,6,Next Friday (2000)
2,2.000000,4,Supernova (2000)
3,2.000000,4,Down to You (2000)
4,2.672414,29,Scream 3 (2000)
...,...,...,...
4773,3.500000,1,Jon Stewart Has Left the Building (2015)
4774,4.000000,1,Black Butler: Book of the Atlantic (2017)
4775,3.500000,1,No Game No Life: Zero (2017)
4776,3.500000,1,Flint (2017)


In [10]:
movies = df_ratings.title.tolist()

In [11]:
my_ratings = np.zeros(df_ratings.shape[0])

my_ratings[2700] = 5   # Toy Story 3 (2010)
my_ratings[2609] = 2   # Persuasion (2007)
my_ratings[929]  = 5   # Lord of the Rings: The Return of the King, The
my_ratings[246]  = 5   # Shrek (2001)
my_ratings[2716] = 3   # Inception
my_ratings[1150] = 5   # Incredibles, The (2004)
my_ratings[382]  = 2   # Amelie (Fabuleux destin d'Amélie Poulain, Le)
my_ratings[366]  = 5   # Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
my_ratings[622]  = 5   # Harry Potter and the Chamber of Secrets (2002)
my_ratings[988]  = 3   # Eternal Sunshine of the Spotless Mind (2004)
my_ratings[2925] = 1   # Louis Theroux: Law & Disorder (2008)
my_ratings[2937] = 1   # Nothing to Declare (Rien à déclarer)
my_ratings[793]  = 5   # Pirates of the Caribbean: The Curse of the Black Pearl (2003)

In [12]:
count = 0
for i in range(len(my_ratings)):
    if my_ratings[i] > 0 :
        count += 1
        print('Rated {} for {}'.format(my_ratings[i], movies[i]))

print('~'*10)
print('Number of Movies Rated: {}'.format(count))

Rated 5.0 for Shrek (2001)
Rated 5.0 for Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Rated 2.0 for Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Rated 5.0 for Harry Potter and the Chamber of Secrets (2002)
Rated 5.0 for Pirates of the Caribbean: The Curse of the Black Pearl (2003)
Rated 5.0 for Lord of the Rings: The Return of the King, The (2003)
Rated 3.0 for Eternal Sunshine of the Spotless Mind (2004)
Rated 5.0 for Incredibles, The (2004)
Rated 2.0 for Persuasion (2007)
Rated 5.0 for Toy Story 3 (2010)
Rated 3.0 for Inception (2010)
Rated 1.0 for Louis Theroux: Law & Disorder (2008)
Rated 1.0 for Nothing to Declare (Rien à déclarer) (2010)
~~~~~~~~~~
Number of Movies Rated: 13


In [13]:
# np.c_ to concatenate horizontally
Y_new = np.c_[my_ratings, Y]  # Add new user ratings to Y, my_ratings is the first column
R_new = np.c_[(my_ratings != 0).astype(int), R]  # Add new user indicator matrix to R

Y_new.shape, R_new.shape

((4778, 444), (4778, 444))

# Preprocessing

In [14]:
total_rating_each_movie = np.sum(R_new * Y_new, axis=1) # axis=1 means the sum is performed along each row
n_viewers_each_movie = np.sum(R_new, axis=1) + 1e-12  # to avoid divided by 0

avg_rating_each_movie = total_rating_each_movie / n_viewers_each_movie
avg_rating_each_movie.shape

(4778,)

In [15]:
Y_normalized = Y_new - (R_new * avg_rating_each_movie.reshape(-1, 1))  # reshape allows for broadcasting
Y_normalized.shape

(4778, 444)

# Training

In [16]:
n_movies, n_users = Y_normalized.shape
n_features = 50

In [17]:
def initialize_parameters(n_movies, n_users, n_features):
    stddev = 1.0 / np.sqrt(n_features)
    
    X = tf.Variable(tf.random.normal((n_movies, n_features), mean=0, stddev=stddev, dtype=tf.float64), name='X')
    W = tf.Variable(tf.random.normal((n_users, n_features), mean=0, stddev=stddev, dtype=tf.float64), name='W')
    B = tf.Variable(tf.zeros((1, n_users), dtype=tf.float64), name='B')
    
    return X, W, B

In [18]:
X, W, B = initialize_parameters(n_movies, n_users, n_features)

X.shape, W.shape, B.shape

(TensorShape([4778, 50]), TensorShape([444, 50]), TensorShape([1, 444]))

In [19]:
lambda_ = 0.01
optimizer = keras.optimizers.Adam(learning_rate=0.1)
epochs = 300

for i in range(0, epochs+1):
    
    # tf.GradientTape() records operations on tensors to compute gradients for backpropagation.
    with tf.GradientTape() as tape:
        cost_value = compute_cost_vectorized(X, W, B, Y_normalized, R_new, lambda_)
        
    # This calculates the gradient of the cost_value with respect to the parameters (X, W, B)
    gradient = tape.gradient(cost_value, [X, W, B])
    
    # The gradients are applied to update the parameters using the chosen optimizer
    optimizer.apply_gradients(zip(gradient, [X, W, B]))

    if i % 10 == 0:
        print(f'Loss at epoch {i:<3}: {cost_value:.02f}')

Loss at epoch 0  : 14830.47
Loss at epoch 10 : 1556.27
Loss at epoch 20 : 644.28
Loss at epoch 30 : 350.08
Loss at epoch 40 : 226.05
Loss at epoch 50 : 165.26
Loss at epoch 60 : 131.17
Loss at epoch 70 : 109.43
Loss at epoch 80 : 94.19
Loss at epoch 90 : 82.63
Loss at epoch 100: 73.42
Loss at epoch 110: 65.84
Loss at epoch 120: 59.50
Loss at epoch 130: 54.23
Loss at epoch 140: 49.83
Loss at epoch 150: 46.80
Loss at epoch 160: 43.68
Loss at epoch 170: 40.78
Loss at epoch 180: 39.04
Loss at epoch 190: 38.59
Loss at epoch 200: 36.04
Loss at epoch 210: 34.95
Loss at epoch 220: 33.70
Loss at epoch 230: 32.94
Loss at epoch 240: 31.89
Loss at epoch 250: 32.17
Loss at epoch 260: 32.94
Loss at epoch 270: 30.36
Loss at epoch 280: 31.54
Loss at epoch 290: 29.64
Loss at epoch 300: 30.48


# Prediction

In [20]:
def predict_ratings(X, W, B):
    norm_ratings_pred = tf.matmul(X, tf.transpose(W)) + B
    ratings_pred = norm_ratings_pred + avg_rating_each_movie.reshape(-1, 1)
    return ratings_pred

In [21]:
ratings_pred = predict_ratings(X, W, B)
ratings_pred

<tf.Tensor: shape=(4778, 444), dtype=float64, numpy=
array([[2.88929009, 4.23400972, 3.60203966, ..., 3.44702118, 4.11380902,
        3.4835261 ],
       [2.96517178, 4.25672255, 3.36178318, ..., 2.92860438, 3.23911706,
        3.39165424],
       [1.41528223, 2.72243757, 2.19817033, ..., 1.87050137, 2.07430863,
        2.02118397],
       ...,
       [3.225211  , 4.41584881, 3.66584232, ..., 3.46899988, 3.82868176,
        3.6168031 ],
       [3.22757754, 4.41620876, 3.66582033, ..., 3.46589651, 3.82751038,
        3.61524618],
       [3.22687864, 4.41605418, 3.66585523, ..., 3.46583929, 3.82813841,
        3.61373107]])>

In [22]:
my_ratings_pred = ratings_pred[:, 0].numpy()
my_ratings_pred.shape

(4778,)

In [23]:
df_my_ratings = pd.concat([pd.Series(my_ratings_pred), pd.Series(my_ratings), df_ratings], axis=1)

df_my_ratings = df_my_ratings.rename(
    columns={df_my_ratings.columns[0]: 'my_ratings_pred', df_my_ratings.columns[1]: 'my_ratings'}
)

df_my_ratings

,my_ratings_pred,my_ratings,mean rating,number of ratings,title
0,2.889290,0.0,3.400000,5,"Yards, The (2000)"
1,2.965172,0.0,3.250000,6,Next Friday (2000)
2,1.415282,0.0,2.000000,4,Supernova (2000)
3,2.115165,0.0,2.000000,4,Down to You (2000)
4,3.063069,0.0,2.672414,29,Scream 3 (2000)
...,...,...,...,...,...
4773,3.225971,0.0,3.500000,1,Jon Stewart Has Left the Building (2015)
4774,3.724452,0.0,4.000000,1,Black Butler: Book of the Atlantic (2017)
4775,3.225211,0.0,3.500000,1,No Game No Life: Zero (2017)
4776,3.227578,0.0,3.500000,1,Flint (2017)


In [24]:
df_my_ratings[df_my_ratings.my_ratings != 0]

,my_ratings_pred,my_ratings,mean rating,number of ratings,title
246,4.999390,5.0,3.867647,170,Shrek (2001)
366,4.999881,5.0,3.761682,107,Harry Potter and the Sorcerer's Stone (a.k.a. ...
382,2.002533,2.0,4.183333,120,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ..."
622,4.999461,5.0,3.598039,102,Harry Potter and the Chamber of Secrets (2002)
793,4.997801,5.0,3.778523,149,Pirates of the Caribbean: The Curse of the Bla...
929,4.998886,5.0,4.118919,185,"Lord of the Rings: The Return of the King, The..."
988,3.002211,3.0,4.160305,131,Eternal Sunshine of the Spotless Mind (2004)
1150,4.997137,5.0,3.836000,125,"Incredibles, The (2004)"
2609,2.001995,2.0,3.333333,3,Persuasion (2007)
2700,4.999265,5.0,4.109091,55,Toy Story 3 (2010)


# Recommendation

In [25]:
movies_not_rated = df_my_ratings[df_my_ratings.my_ratings == 0]

movies_recommended = movies_not_rated.sort_values(by=['my_ratings_pred', 'mean rating'], ascending=False)

movies_recommended.head(10)

,my_ratings_pred,my_ratings,mean rating,number of ratings,title
2145,5.323411,0.0,3.824468,94,Iron Man (2008)
2019,5.155113,0.0,3.898438,64,No Country for Old Men (2007)
51,5.062432,0.0,3.938235,170,Gladiator (2000)
2967,5.058843,0.0,3.910000,50,Harry Potter and the Deathly Hallows: Part 2 (...
72,4.963447,0.0,3.448529,68,"Patriot, The (2000)"
2804,4.957624,0.0,3.989362,47,Harry Potter and the Deathly Hallows: Part 1 (...
3703,4.924713,0.0,5.000000,1,Colourful (Karafuru) (2010)
1930,4.919116,0.0,3.862069,58,Harry Potter and the Order of the Phoenix (2007)
2649,4.902193,0.0,3.943396,53,How to Train Your Dragon (2010)
4064,4.835859,0.0,5.000000,1,My Love (2006)


# Finding Similar Movies (using Euclidean Distance)

In [26]:
def compute_distance_matrix(X, mask_lower_triangle=True):
    """
    Compute the Euclidean distance matrix between all points in X.
    
    Parameters
    ----------
    X : np.ndarray
        Input data matrix of shape (n_movies, n_features). Each row represents a point.
        
    mask_lower_triangle : bool, default=True
        If True, sets the lower triangular part (including the diagonal) of the distance matrix to np.inf.
        This is useful for algorithms that only need the upper triangular distances (e.g., nearest neighbors).
    
    Returns
    -------
    np.ndarray
        A square matrix of shape (n_samples, n_samples), where entry (i, j) contains the Euclidean
        distance between X[i] and X[j]. If mask_lower_triangle=True, the lower triangle is filled with np.inf.
    
    Notes
    -----
    - This implementation is fully vectorized using NumPy, so it is efficient for large datasets.
    - The Euclidean distance is computed using the formula:
    
        ||x_i - x_j||^2 = ||x_i||^2 + ||x_j||^2 - 2 * (x_i · x_j)
        
      Then the square root is taken to get the actual distance.
    """
    
    length_squared = np.sum(np.square(X), axis=1)  # shape of (n_movies, )
    dot_product = np.dot(X, X.T)  # shape of (n_movies, n_movies)

    squared_distance_matrix = length_squared[:, np.newaxis] + length_squared[np.newaxis, :] - 2 * dot_product  # np.newaxis for broadcasting
    distance_matrix = np.sqrt(np.abs(squared_distance_matrix))

    if mask_lower_triangle:
        mask = np.tril(np.ones_like(distance_matrix, dtype=bool), k=0)
        distance_matrix[mask] = np.inf

    return distance_matrix

In [27]:
distance_matrix = compute_distance_matrix(X.numpy())

df_distance_matrix = pd.DataFrame(distance_matrix, columns=None).round(2)
df_distance_matrix

,0,1,2,3,4,5,6,7,8,9,...,4768,4769,4770,4771,4772,4773,4774,4775,4776,4777
0,inf,1.15,1.20,1.42,2.35,1.92,0.94,2.18,1.30,1.19,...,0.86,0.86,0.86,0.86,0.86,0.86,0.86,0.86,0.86,0.86
1,inf,inf,1.07,1.31,2.37,1.62,0.85,2.18,1.38,1.25,...,0.83,0.83,0.83,0.83,0.83,0.83,0.83,0.83,0.83,0.83
2,inf,inf,inf,1.44,2.39,1.95,0.88,1.96,1.39,1.20,...,0.88,0.88,0.88,0.88,0.88,0.88,0.88,0.88,0.88,0.88
3,inf,inf,inf,inf,2.50,1.89,1.07,2.14,1.09,1.50,...,1.07,1.07,1.07,1.07,1.07,1.07,1.07,1.07,1.07,1.07
4,inf,inf,inf,inf,inf,2.46,2.21,3.03,2.23,2.55,...,2.23,2.23,2.23,2.23,2.23,2.23,2.23,2.23,2.23,2.23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4773,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,...,inf,inf,inf,inf,inf,inf,0.00,0.00,0.01,0.01
4774,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,...,inf,inf,inf,inf,inf,inf,inf,0.00,0.01,0.01
4775,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,...,inf,inf,inf,inf,inf,inf,inf,inf,0.01,0.01
4776,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,...,inf,inf,inf,inf,inf,inf,inf,inf,inf,0.00


In [28]:
def find_similar_items(X, k=3):
    """
    Find the k most similar items for each item in X based on Euclidean distance.

    Parameters
    ----------
    X : np.ndarray
        Data matrix of shape (n_movies, n_features).
    k : int, default=3
        Number of similar items to return for each item.

    Returns
    -------
    similar_indices : np.ndarray
        Array of shape (n_movies, k) containing indices of the k nearest neighbors for each item.
    similar_distances : np.ndarray
        Array of shape (n_movies, k) containing distances to the k nearest neighbors.
    """
    
    # Compute the distance matrix (upper triangle filled, lower = inf)
    distance_matrix = compute_distance_matrix(X, mask_lower_triangle=False)
    
    n_movies = X.shape[0]
    similar_indices = np.zeros((n_movies, k), dtype=int)
    similar_distances = np.zeros((n_movies, k), dtype=float)
    
    for i in range(n_movies):
        # Exclude self-distance by setting it to inf
        distances = distance_matrix[i].copy()
        distances[i] = np.inf
        
        # Get k smallest distances
        smallest_indices = np.argpartition(distances, k)[:k]  # returns indices of k smallest element
        sorted_smallest_distances = smallest_indices[np.argsort(distances[smallest_indices])]  # sort those indices by distance
       
        similar_indices[i] = sorted_smallest_distances
        similar_distances[i] = distances[sorted_smallest_distances]
    
    return similar_indices, similar_distances

In [29]:
item_pairs, _ = find_similar_items(X.numpy(), k=5)
item_pairs

array([[ 308, 4031, 1632, 4022, 3311],
       [1264,   67, 1493, 3393, 4517],
       [  15,  272, 1047, 4745, 1171],
       ...,
       [4664, 4773, 4641, 3954, 4774],
       [4588, 4723, 4475, 4771, 4731],
       [4476, 4475, 4304, 4723, 4776]])

In [30]:
def build_similar_movies_df(similar_indices, df_movies, k=5):
    """
    Build a DataFrame showing similar movies for each movie.

    Parameters
    ----------
    similar_indices : np.ndarray
        Output from find_similar_items, shape (n_movies, k)
    df_movies : pd.DataFrame
        DataFrame containing movie information, must have 'title' column
    k : int
        Number of similar items per movie

    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
        - 'title' : main movie title
        - 'similar_title_1', ..., 'similar_title_k' : titles of k nearest movies
    """
    rows = []

    n_movies = similar_indices.shape[0]

    for anchor_id in range(n_movies):
        title = df_movies.iloc[anchor_id]['title']
        row = {'title': title}

        for i, similar_id in enumerate(similar_indices[anchor_id], start=1):
            if i > k:
                break
            similar_title = df_movies.iloc[similar_id]['title']
            row[f'similar_title_{i}'] = similar_title

        rows.append(row)

    return pd.DataFrame(rows)

In [31]:
build_similar_movies_df(item_pairs, df_ratings).head(10)

,title,similar_title_1,similar_title_2,similar_title_3,similar_title_4,similar_title_5
0,"Yards, The (2000)",Ghosts of Mars (2001),Paul Blart: Mall Cop 2 (2015),"Sentinel, The (2006)",Last Knights (2015),Lincoln (2012)
1,Next Friday (2000),Mean Creek (2004),Boys and Girls (2000),North Country (2005),Olympus Has Fallen (2013),The True Memoirs of an International Assassin ...
2,Supernova (2000),Reindeer Games (2000),Kiss of the Dragon (2001),Raising Helen (2004),Jurassic World: Fallen Kingdom (2018),Control Room (2004)
3,Down to You (2000),Drowning Mona (2000),Bait (2000),All the Pretty Horses (2000),Summer Catch (2001),Life or Something Like It (2002)
4,Scream 3 (2000),"Sum of All Fears, The (2002)",Hanging Up (2000),"Wolverine, The (2013)",Jarhead (2005),Final Destination 2 (2003)
5,"Boondock Saints, The (2000)",Ong-Bak: The Thai Warrior (Ong Bak) (2003),The Boss Baby (2017),Friends with Money (2006),Lovely & Amazing (2001),View from the Top (2003)
6,Gun Shy (2000),Kill the Messenger (2014),Barney's Version (2010),Dying of the Light (2014),Alan Partridge: Alpha Papa (2013),You Will Meet a Tall Dark Stranger (2010)
7,"Beach, The (2000)",Silence (2016),It's Such a Beautiful Day (2012),Eye in the Sky (2016),Son of God (2014),Love and Other Drugs (2010)
8,Snow Day (2000),All the Pretty Horses (2000),"Next Best Thing, The (2000)",Drowning Mona (2000),Taking Lives (2004),"Crew, The (2000)"
9,"Tigger Movie, The (2000)",It Might Get Loud (2008),"Other Boleyn Girl, The (2008)",Storytelling (2001),"Wind That Shakes the Barley, The (2006)",Lost in Austen (2008)
